# LLM 단독 테스트 노트북

보이스피싱 탐지용 **Gemma LLM만** 독립 실행하여 테스트합니다.

- **LLM만 독립 실행**: PLM 없이 LLM만 로드 후 텍스트 입력
- **원본 응답 전체 출력**: 모델이 생성한 raw 텍스트 확인
- **JSON 파싱 결과 표시**: comprehensive_risk_score, reasoning, key_evidence
- **대화형/배치 모드**: 단일 텍스트, 여러 샘플 일괄 테스트, 대화형 입력

## 1. 환경 설정 및 LLM 로드

### GPU/CUDA 사용 여부 확인 (필수)

**단일 테스트가 10분 이상 걸리면** 대부분 GPU를 쓰지 않고 CPU만 사용 중입니다. 아래 셀을 먼저 실행해 보세요.

In [2]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print("CUDA 사용 가능:", cuda_ok)
if cuda_ok:
    print("GPU 이름:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("주의: GPU 미사용 시 Gemma-2B 추론이 1~2시간 이상 걸릴 수 있습니다.")
    print("해결: PyTorch CUDA 버전 설치 (https://pytorch.org/get-started/locally/)")

CUDA 사용 가능: False
주의: GPU 미사용 시 Gemma-2B 추론이 1~2시간 이상 걸릴 수 있습니다.
해결: PyTorch CUDA 버전 설치 (https://pytorch.org/get-started/locally/)


In [1]:
import sys
import json
import re
from pathlib import Path

# 노트북 위치 기준으로 프로젝트 루트 계산 (inference, system_ko.txt 경로용)
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "finetuning":
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # scripts/finetuning -> scripts -> project root
else:
    PROJECT_ROOT = NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.finetuning import inference

# 시스템 프롬프트 경로 (프로젝트 루트 기준)
SYSTEM_PROMPT_PATH = PROJECT_ROOT / "system_ko.txt"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"system_ko.txt 존재: {SYSTEM_PROMPT_PATH.exists()}")

c:\1.Project\Voice-Phishing-Protector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\1.Project\Voice-Phishing-Protector
system_ko.txt 존재: True


In [2]:
# LLM만 로드 (PLM은 로드하지 않음)
inference.load_llm()
print("LLM 로드 완료.")

2026-02-02 12:40:40,097 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/gemma-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"
2026-02-02 12:40:40,417 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/gemma-2b-it/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-02-02 12:40:40,685 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/gemma-2b-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-02-02 12:40:40,901 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/gemma-2b-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-02-02 12:40:43,880 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/gemma-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"
2026-02-02 12:40:44,114 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/gemma-2b-it/resolve/main/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 164/16

LLM 로드 완료.


## 2. 프롬프트 구성 및 JSON 파싱 헬퍼

In [3]:
def load_system_prompt(path: Path) -> str:
    """시스템 프롬프트 파일을 읽어 반환합니다."""
    if path.exists():
        return path.read_text(encoding="utf-8")
    return ""

def build_prompt(conversation_text: str, system_prompt_path: Path = None) -> str:
    """
    Gemma용 프롬프트를 구성합니다.
    형식: <start_of_turn>user\n{내용}<end_of_turn>\n<start_of_turn>model\n
    """
    path = system_prompt_path or SYSTEM_PROMPT_PATH
    sys_prompt = load_system_prompt(path)
    user_content = (
        f"{sys_prompt}\n\n### 분석할 대화:\n{conversation_text}\n\n"
        "위 대화를 분석하고 JSON 형식으로만 응답하세요."
    )
    return f"<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

def parse_llm_json(raw_text: str) -> dict:
    """
    LLM 원본 응답에서 JSON을 추출하여 파싱합니다.
    Returns: {"comprehensive_risk_score", "reasoning", "key_evidence", "parse_ok", "raw"}
    """
    clean = raw_text.strip()
    for prefix in ("```json", "```"):
        if clean.startswith(prefix):
            clean = clean[len(prefix):].strip()
    if clean.endswith("```"):
        clean = clean[:-3].strip()
    
    result = {
        "comprehensive_risk_score": 0.0,
        "reasoning": "",
        "key_evidence": [],
        "parse_ok": False,
        "raw": raw_text,
    }
    
    # 방법 1: 전체 JSON
    if clean.startswith("{"):
        try:
            parsed = json.loads(clean)
            result["comprehensive_risk_score"] = float(parsed.get("comprehensive_risk_score", 0.0))
            result["reasoning"] = parsed.get("reasoning", "")
            result["key_evidence"] = parsed.get("key_evidence", []) or []
            result["parse_ok"] = True
            return result
        except json.JSONDecodeError:
            pass
    
    # 방법 2: 정규식으로 JSON 블록 추출
    m = re.search(r'\{[^{}]*"comprehensive_risk_score"[^{}]*\}', clean, flags=re.S)
    if m:
        try:
            parsed = json.loads(m.group(0))
            result["comprehensive_risk_score"] = float(parsed.get("comprehensive_risk_score", 0.0))
            result["reasoning"] = parsed.get("reasoning", "")
            result["key_evidence"] = parsed.get("key_evidence", []) or []
            result["parse_ok"] = True
            return result
        except json.JSONDecodeError:
            pass
    
    # 방법 3: 점수/이유만 추출
    score_m = re.search(r'"comprehensive_risk_score"\s*:\s*([\d.]+)', clean)
    if score_m:
        result["comprehensive_risk_score"] = float(score_m.group(1))
    reason_m = re.search(r'"reasoning"\s*:\s*"([^"]*)"', clean)
    if reason_m:
        result["reasoning"] = reason_m.group(1)
    
    return result

## 3. 단일 텍스트 테스트 (Single)

In [4]:
def test_single(text: str, max_new_tokens: int = 256) -> dict:
    """
    단일 대화 텍스트로 LLM을 호출하고, 원본 응답 + 파싱 결과를 반환합니다.
    """
    prompt = build_prompt(text)
    raw_response = inference.generate_llm(prompt, max_new_tokens=max_new_tokens)
    parsed = parse_llm_json(raw_response)
    parsed["input_text"] = text
    return parsed

# 예시: 아래 변수에 테스트할 대화 텍스트를 넣고 셀 실행
SAMPLE_TEXT = "안녕하세요. 서울중앙지검 수사관입니다. 대포통장이 개설되어 범죄에 연루되었습니다. 안전 계좌로 즉시 이체해 주세요."

result = test_single(SAMPLE_TEXT)
print("=" * 60)
print("[입력 텍스트]")
print(result["input_text"])
print("=" * 60)
print("[원본 응답 전체]")
print(result["raw"])
print("=" * 60)
print("[JSON 파싱 결과]")
print(f"  parse_ok: {result['parse_ok']}")
print(f"  comprehensive_risk_score: {result['comprehensive_risk_score']}")
print(f"  reasoning: {result['reasoning']}")
print(f"  key_evidence: {result['key_evidence']}")

KeyboardInterrupt: 

## 4. 배치 모드 (Batch)

In [ ]:
# 테스트할 샘플 리스트
BATCH_SAMPLES = [
    "안녕하세요. 택배 배송 관련해서 연락드렸습니다. 주소 확인 부탁드립니다.",
    "검찰청입니다. 귀하 계좌가 사기 사건에 연루되었습니다. 안전한 계좌로 이체해 주세요.",
    "고객님, 포인트가 만료 예정입니다. 링크 클릭해서 사용해 주세요.",
]

batch_results = []
for i, text in enumerate(BATCH_SAMPLES):
    print(f"--- 배치 {i+1}/{len(BATCH_SAMPLES)} ---")
    res = test_single(text)
    batch_results.append(res)
    print(f"  score: {res['comprehensive_risk_score']:.3f}, parse_ok: {res['parse_ok']}")
    print()

In [ ]:
# 배치 결과 요약 테이블
import pandas as pd

rows = []
for r in batch_results:
    rows.append({
        "입력(앞 30자)": (r["input_text"][:30] + "...") if len(r["input_text"]) > 30 else r["input_text"],
        "score": r["comprehensive_risk_score"],
        "parse_ok": r["parse_ok"],
        "reasoning": (r["reasoning"][:40] + "...") if len(r["reasoning"]) > 40 else r["reasoning"],
    })
pd.DataFrame(rows)

## 5. 대화형 모드 (Interactive)

In [ ]:
def run_interactive(max_new_tokens: int = 256):
    """
    셀 실행 후 터미널에서 텍스트를 입력하면 LLM 결과를 출력합니다.
    빈 입력으로 엔터 시 종료.
    """
    while True:
        text = input("대화 텍스트 입력 (빈칸 엔터 시 종료): ").strip()
        if not text:
            print("종료합니다.")
            break
        result = test_single(text, max_new_tokens=max_new_tokens)
        print("\n[원본 응답]")
        print(result["raw"])
        print("\n[파싱 결과] score =", result["comprehensive_risk_score"], ", parse_ok =", result["parse_ok"])
        print("-" * 60)

# 주의: Jupyter에서 input()은 노트북 커널에 입력을 받습니다.
# 아래 셀을 실행한 뒤 상단 입력창에 텍스트를 입력하세요.
run_interactive()

### 대화형 대안: 노트북 셀에서 직접 입력

In [ ]:
# 이 셀에서 MY_INPUT 에 원하는 텍스트를 넣고 실행하면 됩니다.
MY_INPUT = "여기에 테스트할 대화 내용을 입력하세요."

if MY_INPUT.strip():
    r = test_single(MY_INPUT)
    print("=" * 60)
    print("[원본 응답 전체]")
    print(r["raw"])
    print("=" * 60)
    print("[JSON 파싱 결과]")
    print("  comprehensive_risk_score:", r["comprehensive_risk_score"])
    print("  reasoning:", r["reasoning"])
    print("  key_evidence:", r["key_evidence"])
    print("  parse_ok:", r["parse_ok"])
else:
    print("MY_INPUT 을 채운 뒤 셀을 실행하세요.")